<a href="https://colab.research.google.com/github/catchshashank/optimal-stopping/blob/main/notebooks/optimal-stopping-llama%203.2-run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Llama 3.2 1B + Behavioral Cloning**

Replicated by [Shashank Dubey](https://github.com/catchshashank) to the [stopping-agents](https://github.com/catchshashank/optimal-stopping) repository.

In [2]:
# ── Colab / cloud setup ───────────────────────────────────────────────────────────────────────
# Run this cell first. It is a no-op when running locally.
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.46.0",
        "datasets>=2.18.0",
        "huggingface_hub>=0.22.0",
        "accelerate>=0.27.0",
        "scikit-learn>=1.3.0",
        "tqdm>=4.66.0",
        "openai>=1.0.0",
    ], check=True)

    # Mount Google Drive to persist checkpoints across sessions
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/optimal-stopping"
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

    # Load HF token from Colab Secrets (key icon in left sidebar, name: HF_TOKEN)
    # or you will be prompted to paste it below.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab secrets.")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token: ")

else:
    DRIVE_CHECKPOINT_DIR = None
    print("Running locally — skipping Colab setup.")


Mounted at /content/drive
Paste your Hugging Face token: ··········


#### **Import Dependencies:** Call the core libraries
---

- **🤗 Datasets / Transformers / Hub**: loading data, tokenization, model + training
- **PyTorch**: tensor ops, log-softmax, GPU execution
- **pandas / numpy**: building conversation-level tables and features
- **sklearn**: splitting and ROC-AUC evaluation
- **tqdm**: progress bars for batched inference

In [3]:
import datasets
import huggingface_hub # needed for Llama models
import math
import numpy as np
import pandas as pd
import torch
import transformers

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

#### **Global parameters:** Defines the optimal-stopping objective
---
- `COST_PER_UNIT_TIME`: penalty per second spent on the call
- `BENEFIT_PER_POSITIVE_OUTCOME`: reward if the call ends in a sale
- `DECISION_OPPORTUNITIES`: times (seconds) at which the agent may choose **quit** vs **continue** (here: 45s and 60s).

In [4]:
import os

HF_TOKEN      = os.environ.get("HF_TOKEN",      "HF_TOKEN")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "OPENAI_API_KEY")

COST_PER_UNIT_TIME = 0.1
BENEFIT_PER_POSITIVE_OUTCOME = 10.0
DECISION_OPPORTUNITIES = [45, 60]  # time in seconds at which the
                                   # agent can decide to quit or wait
                                   # code is tailored to just 2 right now

# ── Backbone selection ──────────────────────────────────────────────────────────────────────────────
def _output_dir(name):
    """Returns Drive path in Colab, local path otherwise."""
    if DRIVE_CHECKPOINT_DIR:
        return os.path.join(DRIVE_CHECKPOINT_DIR, name)
    return f"./{name}/"

BACKBONE_CONFIGS = {
    "llama-3.2-3b": {
        "type": "hf",
        "hf_id": "meta-llama/Llama-3.2-3B",  # base model, not instruction-tuned
        "output_dir": _output_dir("llama-3.2-3B"),
    },
    "mistral-7b": {
        "type": "hf",
        "hf_id": "mistralai/Mistral-7B-v0.1",
        "output_dir": _output_dir("mistral-7B"),
    },
    "gpt-4o": {
        "type": "openai",
        "openai_id": "gpt-4o",
    },
}

BACKBONE = "llama-3.2-3b"  # ← change this to switch backbone
MODEL_CONFIG = BACKBONE_CONFIGS[BACKBONE]


#### **Load the dataset and create labels**
---
- Loads the synthetic sales conversation CSV.
- Creates `is_sale` (`sale`=1, `no sale`=0).
- Computes `duration` per conversation as `max(end_time)`.

In [5]:
dataset_url = "/content/synthetic_sales_conversations(1).csv"

diarized_conversations = pd.read_csv(dataset_url)

# Add column ("is_sale or not")
diarized_conversations["is_sale"] =\
        diarized_conversations["outcome"].apply(
            lambda x: 1 if x == "sale" else 0 if x == "no sale" else np.nan)

# Add column ("duration")
diarized_conversations["duration"] =\
    diarized_conversations.groupby("conversation_id")["end_time"].transform("max")

diarized_conversations.head()

,conversation_id,speaker_id,start_time,end_time,text,outcome,is_sale,duration
0,20756_1,0,1.25,6.03,"Hello, is this Mr. Harris? My name is Leah fro...",no sale,0,62.07
1,20756_1,1,6.36,7.59,"Yes, speaking. I’m alright, thanks. Can I ask ...",no sale,0,62.07
2,20756_1,0,7.98,12.84,"Of course, thanks for asking. I’m reaching out...",no sale,0,62.07
3,20756_1,1,13.14,15.50,Alright… I guess I can listen for a minute.,no sale,0,62.07
4,20756_1,0,15.89,22.14,"Thank you! So, our new BrightSaver plan locks ...",no sale,0,62.07


#### **Train, val, and test split at the conversation level**
---
Splits by **conversation_id** (not turn rows) to avoid leakage.
Uses `stratify` on `is_sale` to keep label proportions similar across splits.

In [6]:
all_conversation_ids =\
    diarized_conversations[["conversation_id", "is_sale"]].drop_duplicates()["conversation_id"].values
all_outcomes =\
    diarized_conversations[["conversation_id", "is_sale"]].drop_duplicates()["is_sale"].values

# Split dataset into train + test datasets ensuring class balance
train_conversation_ids, test_conversation_ids, train_outcomes, test_outcomes =\
    train_test_split(all_conversation_ids, all_outcomes, test_size=0.25, random_state=42,
                     stratify=all_outcomes)

# Further split training data into train + val dataset
train_conversation_ids, val_conversation_ids, train_outcomes, val_outcomes =\
    train_test_split(train_conversation_ids, train_outcomes, test_size=0.25, random_state=42,
                     stratify=train_outcomes)

# Final diarized conversation datasets = train + val + test (25%)
diarized_conversations_train =\
    diarized_conversations[diarized_conversations["conversation_id"].isin(train_conversation_ids)]
diarized_conversations_val =\
    diarized_conversations[diarized_conversations["conversation_id"].isin(val_conversation_ids)]
diarized_conversations_test =\
    diarized_conversations[diarized_conversations["conversation_id"].isin(test_conversation_ids)]

print(len(diarized_conversations_train), "train conversations.")
print(len(diarized_conversations_val), "validation conversations.")
print(len(diarized_conversations_test), "test conversations.")

1903 train conversations.
651 validation conversations.
860 test conversations.


#### **Build partial transcripts at each decision time**
---
Creates `transcript_speaker_45` and `transcript_speaker_60` by:
1) formatting each utterance as `Speaker X: text`
2) keeping turns with `end_time < m`
3) concatenating to a single transcript per conversation.

In [7]:
m1, m2 = sorted(DECISION_OPPORTUNITIES)

# Loop through train, val, and test data and store them in data_transcripts
data_transcripts = {}
for df, dftype in zip([diarized_conversations_train,
                       diarized_conversations_val,
                       diarized_conversations_test],
                      ["train", "val", "test"]):

    data_transcripts[dftype] = df.copy()

    data_transcripts[dftype]["transcript"] =\
        "Speaker " +\
        data_transcripts[dftype]["speaker_id"].astype(str) + ": " +\
        data_transcripts[dftype]["text"]

    # Inner loop dictionary to hold transcripts at decision points
    transcripts = {}
    for m in [m1, m2]:
        transcripts[m] =\
            data_transcripts[dftype].loc[(data_transcripts[dftype]["end_time"]>=0) & # Select only transcripts from 0 to m seconds
                                         (data_transcripts[dftype]["end_time"]<m)]\
                    .groupby("conversation_id")["transcript"]\
                    .apply(lambda x: '\n'.join(x))\
                    .reset_index(name="transcript_speaker_" + str(m)) # Block name = transcript_speaker_m (=45/60)

    # Build mainframe with conversation_id, duration, is_sale, and transcript (at 'm')
    data_transcripts[dftype] = \
        pd.merge(data_transcripts[dftype][["conversation_id",
                                   "duration",
                                   "is_sale"]].drop_duplicates(),
                 transcripts[m1],
                 on="conversation_id", how="left", validate="one_to_one")\
                    .merge(transcripts[m2],
                           on="conversation_id", how="left", validate="one_to_one")

print(len(data_transcripts["train"]), "train conversations.")
print(len(data_transcripts["val"]), "validation conversations.")
print(len(data_transcripts["test"]), "test conversations.")

# Check Data Integrity
assert len(data_transcripts["train"]) +\
         len(data_transcripts["val"]) +\
         len(data_transcripts["test"]) == diarized_conversations["conversation_id"].nunique()

data_transcripts["train"].head()

112 train conversations.
38 validation conversations.
50 test conversations.


,conversation_id,duration,is_sale,transcript_speaker_45,transcript_speaker_60
0,20756_1,62.07,0,"Speaker 0: Hello, is this Mr. Harris? My name ...","Speaker 0: Hello, is this Mr. Harris? My name ..."
1,59321_6,55.37,0,Speaker 0: Good afternoon! Is this Ms. Parker?...,Speaker 0: Good afternoon! Is this Ms. Parker?...
2,92837_7,43.64,0,"Speaker 0: Good afternoon, may I speak with Mr...","Speaker 0: Good afternoon, may I speak with Mr..."
3,58241_9,50.01,0,"Speaker 0: Hello, may I speak with Ms. Jenkins...","Speaker 0: Hello, may I speak with Ms. Jenkins..."
4,20567_11,45.19,0,"Speaker 0: Good afternoon, is this Mr. Carver?...","Speaker 0: Good afternoon, is this Mr. Carver?..."


#### **Wrap partial transcripts into prompting “states”**
---
Defines `convert_to_state(transcript, t)` which creates a prompt ending with: “Will this call end in a sale (respond with 'yes' or 'no').” Then constructs `s45` and `s60` for the two decision times.

In [8]:
def convert_to_state(transcript, t):
    assert type(transcript) == str

    # Structured prompt for language model
    state = "Below is the first " + str(t) +\
            " seconds of the sales call between the sales agent Speaker 0 and" +\
            " the customer Speaker 1:\n" +\
            transcript + "\n" +\
            "Will this call end in a sale (respond with 'yes' or 'no'):  "

    return state

for df in [data_transcripts["train"],
           data_transcripts["val"],
           data_transcripts["test"]]:

    # New column (s + m) = The function is called for each row with relevant transcripts + time (m)
    for m in [m1, m2]:
        df.loc[:, "s" + str(m)] = df.apply(lambda x:
                                            convert_to_state(x["transcript_speaker_" + str(m)],
                                                             m), axis=1)

print("Example state at", m1, "seconds:")
print(data_transcripts["train"]["s" + str(m1)].values[0])
print()
print("Example state at", m2, "seconds:")
print(data_transcripts["train"]["s" + str(m2)].values[0])

Example state at 45 seconds:
Below is the first 45 seconds of the sales call between the sales agent Speaker 0 and the customer Speaker 1:
Speaker 0: Hello, is this Mr. Harris? My name is Leah from Sunview Energy—how are you today?
Speaker 1: Yes, speaking. I’m alright, thanks. Can I ask what this is about?
Speaker 0: Of course, thanks for asking. I’m reaching out because we’re offering a new energy plan that could qualify you for a 15% discount on your electric bill. I wanted to see if I could quickly tell you about it.
Speaker 1: Alright… I guess I can listen for a minute.
Speaker 0: Thank you! So, our new BrightSaver plan locks in your rate for twelve months—there’s no change in price based on the time of day, and there are no hidden fees. And for this month, you’d also get an automatic 15% off your supply charges.
Speaker 1: Is this something I have to switch providers for? I’m pretty happy with who I have now.
Speaker 0: You would stay connected to your local utility for service a

#### **Construct optimal state-action pairs**
---
1) Given the `COST_PER_UNIT_TIME` and `BENEFIT_PER_POSITIVE_OUTCOME`, the code maps the conversation transcript before each decision opportunity to the optimal action (`wait` or `quit`) to take in that state.
2) The optimal action is the one that maximizes the cumulative reward.

In [10]:
optimal_state_action_pairs = {}
for dftype, df in data_transcripts.items():

    # Stop at m1
    df["rq" + str(m1)] = -m1 * COST_PER_UNIT_TIME # stop at 30

    # Continue at m1, Stop at m2
    df["rq" + str(m2)] = df["is_sale"].astype(int)\
                        * BENEFIT_PER_POSITIVE_OUTCOME\
                        * (df["duration"]<=m2).astype(int) \
                        - df["duration"].apply(lambda x: min(m2, x)) * COST_PER_UNIT_TIME

    # Never quit
    df["rc" + str(m2)] = df["is_sale"].astype(int) * BENEFIT_PER_POSITIVE_OUTCOME\
                        - df["duration"] * COST_PER_UNIT_TIME

    # Reward Function
    df["max_reward"] = df[["rq" + str(m1), "rq" + str(m2), "rc" + str(m2)]].max(axis=1)

    # Optimal Condition = Quit at m1
    df.loc[df["max_reward"]==df["rq" + str(m1)], "a" + str(m1)] = "no"
    df.loc[df["max_reward"]==df["rq" + str(m1)], "a" + str(m2)] = "no"

    # Optimal Condition = Continue at m1, Quit at m2
    df.loc[df["max_reward"]==df["rq" + str(m2)], "a"  + str(m1)] = "yes"
    df.loc[df["max_reward"]==df["rq" + str(m2)], "a"  + str(m2)] = "no"

    # Optimal Condition = Never Quit
    df.loc[df["max_reward"]==df["rc" + str(m2)], "a" + str(m1)] = "yes"
    df.loc[df["max_reward"]==df["rc" + str(m2)], "a" + str(m2)] = "yes"

    optimal_state_action_pairs[dftype] = []

    # Build new df paired by conversation_id, state, optimal action, and duration
    for m in [m1, m2]:
        optimal_actions = df[df["duration"]>=m]\
                                [["conversation_id", "s" + str(m), "a" + str(m), "is_sale",
                                  "duration"]]
        optimal_actions = optimal_actions.values.tolist()
        optimal_state_action_pairs[dftype].extend(optimal_actions)

    optimal_state_action_pairs[dftype] =\
        pd.DataFrame(optimal_state_action_pairs[dftype],
                     columns=["conversation_id", "state", "action",
                              "is_sale", "duration"])

print("Distribution of actions in the optimal state-action pairs:")
print(optimal_state_action_pairs["train"]["action"].value_counts())
print(optimal_state_action_pairs["val"]["action"].value_counts())
print(optimal_state_action_pairs["test"]["action"].value_counts())
if len(optimal_state_action_pairs["train"]["action"].value_counts()) == 1:
    print("Only one optimal action in the training set; imitation learning will fail.")

optimal_state_action_pairs["train"].head()

Distribution of actions in the optimal state-action pairs:
action
yes    108
no      67
Name: count, dtype: int64
action
yes    38
no     22
Name: count, dtype: int64
action
yes    49
no     30
Name: count, dtype: int64


,conversation_id,state,action,is_sale,duration
0,20756_1,Below is the first 45 seconds of the sales cal...,no,0,62.07
1,59321_6,Below is the first 45 seconds of the sales cal...,no,0,55.37
2,58241_9,Below is the first 45 seconds of the sales cal...,no,0,50.01
3,20567_11,Below is the first 45 seconds of the sales cal...,no,0,45.19
4,10523_13,Below is the first 45 seconds of the sales cal...,no,0,65.04


### **Behavioral Cloning**
---
Now we can fine-tune our large language model to generate the optimal action for every state.

We fine-tune using `Trainer` in `Transformers` library instead of `SFTTRainer` in `TRL` for 2 reasons:

   1. By default, `SFTTrainer` calculates the loss for *all* tokens, including ones in the prompt. One way to work around this using the prompt-completion pairs data format.

   2. `SFTTrainer` does not enable custom metrics. We want to report the action predicton validation AUC during training, since it is more correlated with our final goal than the validation loss.

#### **Load base LLM + tokenizer**
---
1) From Hugging Face, we load a Llama causal LM + tokenizer.
2) Set padding configuration (EOS as pad, left-padding for batching).

In [11]:
if MODEL_CONFIG["type"] == "hf":
    huggingface_hub.login(token=HF_TOKEN)
    model_name = MODEL_CONFIG["hf_id"]
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = transformers.AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
        print("WARNING: Resizing the embedding matrix to match the tokenizer vocab size.")
        model.resize_token_embeddings(len(tokenizer))
    openai_client = None

elif MODEL_CONFIG["type"] == "openai":
    import openai
    openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
    model, tokenizer = None, None
    print(f"Using OpenAI model: {MODEL_CONFIG['openai_id']} — no local weights loaded.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

#### **Tokenize + mask prompt tokens in labels**
---
Creates supervised examples `prompt -> completion` where completion is `yes/no`.
`labels` uses `-100` for prompt tokens so loss is computed only on the completion.

In [12]:
if MODEL_CONFIG["type"] == "hf":
    train_dataset = datasets.Dataset.from_dict(
        {"prompt": [state for state in optimal_state_action_pairs["train"]["state"].values],
         "completion": [action.strip()
                        for action in optimal_state_action_pairs["train"]["action"].values]}).shuffle()

    val_dataset = datasets.Dataset.from_dict(
        {"prompt": [state for state in optimal_state_action_pairs["val"]["state"].values],
         "completion": [action.strip()
                        for action in optimal_state_action_pairs["val"]["action"].values]}).shuffle()

    def tokenize_fn(example, add_label):
        # start with the BOS token if it exists
        if tokenizer.bos_token is not None:
            encoded_prompt = tokenizer.encode(tokenizer.bos_token +
                                              example["prompt"],
                                              add_special_tokens=False)
        else:
            encoded_prompt = tokenizer.encode(example["prompt"],
                                              add_special_tokens=False)

        # add the label if needed for the training and validation datasets
        if add_label:
            encoded_label = tokenizer.encode(example["completion"] + tokenizer.eos_token,
                                             add_special_tokens=False)
            return {"input_ids": encoded_prompt + encoded_label,
                    "attention_mask": [1] * (len(encoded_prompt) + len(encoded_label)),
                    "labels": [-100] * len(encoded_prompt) + encoded_label}
        else:
            return {"input_ids": encoded_prompt,
                    "attention_mask": [1] * len(encoded_prompt),
                    "labels": [-100] * len(encoded_prompt)}

    train_dataset = train_dataset.map(tokenize_fn,
                                      remove_columns=["prompt", "completion"],
                                      fn_kwargs={"add_label": True})
    val_dataset = val_dataset.map(tokenize_fn,
                                  remove_columns=["prompt", "completion"],
                                  fn_kwargs={"add_label": True})

    print("Tokenization test:")
    print(tokenizer.decode(train_dataset[0]["input_ids"]))
    print()
    print("Expected Label (action):")
    print(tokenizer.decode([l for l in train_dataset[0]["labels"] if l != -100]))


Map:   0%|          | 0/175 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenization test:
<|begin_of_text|>Below is the first 45 seconds of the sales call between the sales agent Speaker 0 and the customer Speaker 1:
Speaker 0: Good afternoon! Is this Ms. Rivera? Hi, this is Mark from Northwind Energy Solutions.
Speaker 1: Yes, speaking. How can I help you?
Speaker 0: Thanks so much for taking my call! I'm reaching out because you've been a loyal customer with us for three years, and we're rolling out a new energy plan that comes bundled with special discounts for accounts like yours.
Speaker 1: That sounds interesting. Can you tell me more about the plan?
Speaker 0: Absolutely! So, our new plan locks in your electricity rate at 12.5 cents per kilowatt hour for the next 12 months, plus you get a $7 discount on your monthly energy bill just for enrolling.
Speaker 0: Additionally, we’ll give you a one-time $30 bill credit if you sign up before the end of the week.
Speaker 1: So I'd be saving on my monthly bill and get a credit upfront?
Speaker 0: Exactly, p

#### **Fine-tuning with `Trainer` + ROC-AUC**
---
Defines `compute_metrics` to compute AUC for YES vs NO from logits, then trains with early stopping and selects the best checkpoint by validation AUC.

In [13]:
if MODEL_CONFIG["type"] == "hf":
    YES_TOKEN_ID = tokenizer.encode("yes", add_special_tokens=False)[-1]
    NO_TOKEN_ID  = tokenizer.encode("no",  add_special_tokens=False)[-1]

    def preprocess_logits_for_metrics(logits, labels):
        """
        Original Trainer may have a memory leak.
        This is a workaround to avoid storing too many tensors that are not needed.
        """
        return logits[:, -3, :]  # last non-padding token logits only, for causal LM

    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        labels = [l[(l != -100) & (l != tokenizer.pad_token_id)][0] for l in labels]

        logprobs = [torch.log_softmax(torch.from_numpy(s), dim=-1).numpy()
                    for s in logits]
        labelprobs = [math.exp(logprob[label]) for logprob, label in zip(logprobs, labels)]

        ytrue = [1 if label == YES_TOKEN_ID else 0 for label in labels]
        ypred = [labelprob if label == YES_TOKEN_ID else 1.0 - labelprob
                 for labelprob, label in zip(labelprobs, labels)]
        auc = roc_auc_score(ytrue, ypred)

        return {"auc": auc}

    training_args = transformers.TrainingArguments(
        output_dir=MODEL_CONFIG["output_dir"],
        remove_unused_columns=False,

        save_strategy="best",
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=1,

        per_device_train_batch_size=10,
        per_device_eval_batch_size=10,
        gradient_accumulation_steps=1,

        num_train_epochs=2,
        learning_rate=1e-4,
        optim="adamw_torch",

        load_best_model_at_end=True,
        metric_for_best_model="auc",
        greater_is_better=True,

        report_to="none",

        bf16=True,   # requires bfloat16-capable GPU (e.g. A100, H100, RTX 3090+)
        fp16=False,
    )

    trainer = transformers.Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset.shuffle(),
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=transformers.data.DataCollatorForSeq2Seq(tokenizer),
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics,
        callbacks=[transformers.EarlyStoppingCallback(early_stopping_patience=1)]
    )

    trainer.train(resume_from_checkpoint=False)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Epoch,Training Loss,Validation Loss,Auc
1,4.738503,0.913390,0.000000
2,0.505996,0.346606,0.090909


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### **Evaluate the model**

#### **Inference: generate yes/no and compute `prob_yes`**
---
Runs batched greedy generation for `s45` and `s60`, and uses generation scores to approximate confidence, then merges predictions back and reports AUC vs `is_sale`.

In [14]:
val_prompts  = list(optimal_state_action_pairs["val"]["state"].values)
test_prompts = list(optimal_state_action_pairs["test"]["state"].values)


def _run_hf_inference(prompts, inference_model, batch_size=72):
    """Batched greedy inference with a local HF model; returns (responses, logprobs)."""
    responses, logprobs_out = [], []
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]
        batch = tokenizer(batch_prompts,
                          return_tensors="pt",
                          padding=True,
                          add_special_tokens=True,
                          truncation=True).to("cuda")
        with torch.no_grad():
            outputs = inference_model.generate(
                **batch,
                max_new_tokens=2,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                temperature=None, top_p=None, top_k=None,
                return_dict_in_generate=True, output_scores=True,
            )
        seqs = outputs.sequences
        prompt_len = batch["input_ids"].shape[1]
        generated_tokens = seqs[:, prompt_len:]
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        responses.extend([d.strip().lower() for d in decoded])
        scores = outputs.scores
        lps = [torch.log_softmax(s, dim=-1) for s in scores]
        lps = lps[0][torch.arange(lps[0].size(0)), seqs[:, -2].view(-1)]
        logprobs_out.extend(lps.cpu().numpy())
    return responses, logprobs_out


def _prob_yes_from_top_logprobs(top_logprobs):
    """Normalise OpenAI top-logprobs to a probability over {yes, no}."""
    lp_map = {entry.token.strip().lower(): entry.logprob for entry in top_logprobs}
    p_yes = math.exp(lp_map.get("yes", -100.0))
    p_no  = math.exp(lp_map.get("no",  -100.0))
    total = p_yes + p_no
    return p_yes / total if total > 0 else 0.5


def _run_openai_inference(prompts):
    """Call the OpenAI chat API for each prompt; returns a list of prob_yes values."""
    prob_yes_list = []
    for prompt in tqdm(prompts):
        response = openai_client.chat.completions.create(
            model=MODEL_CONFIG["openai_id"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1,
            logprobs=True,
            top_logprobs=20,
        )
        top_lps = response.choices[0].logprobs.content[0].top_logprobs
        prob_yes_list.append(_prob_yes_from_top_logprobs(top_lps))
    return prob_yes_list


print("Getting test responses...")
if MODEL_CONFIG["type"] == "hf":
    responses_test, logprobs_test = _run_hf_inference(test_prompts, trainer.model)
    prob_yes_test = None  # derived in the next cell
elif MODEL_CONFIG["type"] == "openai":
    prob_yes_test = _run_openai_inference(test_prompts)
    responses_test = logprobs_test = None

print("Getting validation responses...")
if MODEL_CONFIG["type"] == "hf":
    responses_val, logprobs_val = _run_hf_inference(val_prompts, trainer.model)
    prob_yes_val = None
elif MODEL_CONFIG["type"] == "openai":
    prob_yes_val = _run_openai_inference(val_prompts)
    responses_val = logprobs_val = None


Getting test responses...


  0%|          | 0/2 [00:00<?, ?it/s]

Getting validation responses...


  0%|          | 0/1 [00:00<?, ?it/s]

#### **Store validation and test responses**

In [15]:
if MODEL_CONFIG["type"] == "hf":
    predictions = optimal_state_action_pairs["test"].copy()
    predictions["response"] = responses_test
    predictions["logprob"]  = logprobs_test
    predictions["prob"]     = predictions["logprob"].apply(lambda x: math.exp(x))
    predictions.loc[predictions["response"] == "yes", "prob_yes"] = \
        predictions.loc[predictions["response"] == "yes", "prob"]
    predictions.loc[predictions["response"] != "yes", "prob_yes"] = \
        1.0 - predictions.loc[predictions["response"] != "yes", "prob"]

    predictions_val = optimal_state_action_pairs["val"].copy()
    predictions_val["response"] = responses_val
    predictions_val["logprob"]  = logprobs_val
    predictions_val["prob"]     = predictions_val["logprob"].apply(lambda x: math.exp(x))
    predictions_val.loc[predictions_val["response"] == "yes", "prob_yes"] = \
        predictions_val.loc[predictions_val["response"] == "yes", "prob"]
    predictions_val.loc[predictions_val["response"] != "yes", "prob_yes"] = \
        1.0 - predictions_val.loc[predictions_val["response"] != "yes", "prob"]

elif MODEL_CONFIG["type"] == "openai":
    predictions = optimal_state_action_pairs["test"].copy()
    predictions["prob_yes"] = prob_yes_test

    predictions_val = optimal_state_action_pairs["val"].copy()
    predictions_val["prob_yes"] = prob_yes_val

test_with_predictions = pd.merge(left=data_transcripts["test"],
                                  right=predictions[["conversation_id", "state", "prob_yes"]],
                                  left_on=["conversation_id", "s" + str(m1)],
                                  right_on=["conversation_id", "state"],
                                  how="left", validate="one_to_one")\
                            .drop(columns=["state"])\
                            .rename(columns={"prob_yes": "prob_yes_" + str(m1)})\
                         .merge(right=predictions[["conversation_id", "state", "prob_yes"]],
                                left_on=["conversation_id", "s" + str(m2)],
                                right_on=["conversation_id", "state"],
                                how="left", validate="one_to_one")\
                            .drop(columns=["state"])\
                            .rename(columns={"prob_yes": "prob_yes_" + str(m2)})

test_with_predictions.loc[
    test_with_predictions["prob_yes_" + str(m1)].isnull(), "prob_yes_" + str(m1)] = 0
test_with_predictions.loc[
    test_with_predictions["prob_yes_" + str(m2)].isnull(), "prob_yes_" + str(m2)] = 0

val_with_predictions = pd.merge(left=data_transcripts["val"],
                                 right=predictions_val[["conversation_id", "state", "prob_yes"]],
                                 left_on=["conversation_id", "s" + str(m1)],
                                 right_on=["conversation_id", "state"],
                                 how="left", validate="one_to_one")\
                           .drop(columns=["state"])\
                           .rename(columns={"prob_yes": "prob_yes_" + str(m1)})\
                        .merge(right=predictions_val[["conversation_id", "state", "prob_yes"]],
                               left_on=["conversation_id", "s" + str(m2)],
                               right_on=["conversation_id", "state"],
                               how="left", validate="one_to_one")\
                           .drop(columns=["state"])\
                           .rename(columns={"prob_yes": "prob_yes_" + str(m2)})

val_with_predictions.loc[
    val_with_predictions["prob_yes_" + str(m1)].isnull(), "prob_yes_" + str(m1)] = 0
val_with_predictions.loc[
    val_with_predictions["prob_yes_" + str(m2)].isnull(), "prob_yes_" + str(m2)] = 0

print("Val",  m1, "ROC-AUC: {:.2f}".format(
    roc_auc_score(val_with_predictions["is_sale"].values,
                  val_with_predictions["prob_yes_" + str(m1)].values)))
print("Test", m1, "ROC-AUC: {:.2f}".format(
    roc_auc_score(test_with_predictions["is_sale"].values,
                  test_with_predictions["prob_yes_" + str(m1)].values)))

print("Val",  m2, "ROC-AUC: {:.2f}".format(
    roc_auc_score(val_with_predictions["is_sale"].values,
                  val_with_predictions["prob_yes_" + str(m2)].values)))
print("Test", m2, "ROC-AUC: {:.2f}".format(
    roc_auc_score(test_with_predictions["is_sale"].values,
                  test_with_predictions["prob_yes_" + str(m2)].values)))


Val 45 ROC-AUC: 0.66
Test 45 ROC-AUC: 0.65
Val 60 ROC-AUC: 0.92
Test 60 ROC-AUC: 0.85


#### **Simulate a threshold-based stopping policy**
---
Given thresholds at 45 and 60, simulates quitting/continuing and computes benefit − time cost, returning average reward per conversation.

In [16]:
def simulate_threshold(threshold_m1, threshold_m2, df):
    # quit at m1
    calls_quit_at_m1 = df.loc[(df["prob_yes_" + str(m1)] < threshold_m1)]

    # continue at m1, ended before m2
    calls_continued_at_m1_and_ended = df.loc[(df["prob_yes_" + str(m1)] >= threshold_m1) &
                                             (df["duration"]<m2)]

    # continued at m1, did not end before m2, quit at m2
    calls_continued_at_m1_and_quit_at_m2 = df.loc[(df["prob_yes_" + str(m1)] >= threshold_m1) &
                                                  (df["prob_yes_" + str(m2)] < threshold_m2) &
                                                  (df["duration"]>=m2)]

    # continue at m1, did not end before m2, continued at m2
    calls_continued_at_m2 = df.loc[(df["prob_yes_" + str(m1)] >= threshold_m1) &
                                   (df["prob_yes_" + str(m2)] >= threshold_m2) &
                                   (df["duration"]>=m2)]

    assert len(calls_quit_at_m1) + len(calls_continued_at_m1_and_ended) +\
          len(calls_continued_at_m1_and_quit_at_m2) + \
          len(calls_continued_at_m2) == len(df)

    total_sales = calls_continued_at_m1_and_ended["is_sale"].sum() +\
                    calls_continued_at_m2["is_sale"].sum()
    total_sales_benefit = total_sales * BENEFIT_PER_POSITIVE_OUTCOME

    total_time = (len(calls_quit_at_m1) * m1 +\
                  len(calls_continued_at_m1_and_quit_at_m2) * m2 +\
                    calls_continued_at_m1_and_ended["duration"].sum() +\
                    calls_continued_at_m2["duration"].sum())
    total_cost = total_time * COST_PER_UNIT_TIME

    total_reward = (total_sales_benefit - total_cost)
    average_reward = total_reward/len(df)

    # assert benefit.sum() == total_sales_benefit
    # assert np.isclose(cost.sum(), total_cost)
    # assert np.isclose(reward.sum(), total_reward)

    return total_reward, average_reward, total_sales, total_time

print("Test set reward without the stopping agent:")
total_reward_noagent, average_reward_noagent,\
    total_sales_noagent, total_time_noagent =\
    simulate_threshold(0, 0, test_with_predictions)
print("Total reward on test:", total_reward_noagent)
print("Avg. reward on test:", average_reward_noagent)
print("Total sales on test:", total_sales_noagent)
print("Total time on test (seconds):", total_time_noagent)

assert int(total_sales_noagent) == int(test_with_predictions["is_sale"].sum())
assert total_time_noagent == test_with_predictions["duration"].sum()
assert total_reward_noagent ==\
    (total_sales_noagent * BENEFIT_PER_POSITIVE_OUTCOME - total_time_noagent * COST_PER_UNIT_TIME)
assert average_reward_noagent ==\
    total_reward_noagent / len(test_with_predictions)

Test set reward without the stopping agent:
Total reward on test: -77.97900000000004
Avg. reward on test: -1.5595800000000009
Total sales on test: 25
Total time on test (seconds): 3279.79


#### **Threshold Tuning and Agent Evaluation**
---
1) Perform backward-induction threshold tuning process to find the optimal `prob_yes` thresholds for both decision points (`m1` and `m2`) using the validation set.
2) Apply these optimized thresholds to simulate the stopping agent's performance on the test set, computing rewards, sales, and time metrics.
3) Finally, compare these results against a baseline where no stopping agent is used.


In [17]:
best_threshold_at_m = {}
num_grid_points = 10000

m = m1
prob_column = "prob_yes_" + str(m)
best_reward = -10000000
for candidate_threshold in np.linspace(val_with_predictions[prob_column].min()-10**-12,
                                       val_with_predictions[prob_column].max()+10**-12,
                                       num=num_grid_points):

    total_reward, average_reward, total_sales, total_time =\
        simulate_threshold(0, candidate_threshold, val_with_predictions)

    if average_reward > best_reward:
        best_reward = average_reward
        best_threshold_at_m[m] = candidate_threshold

print("Best threshold at m=" + str(m) + ":", best_threshold_at_m[m])

m = m2
prob_column = "prob_yes_" + str(m)
best_reward = -10000000
for candidate_threshold in np.linspace(val_with_predictions[prob_column].min()-10**-12,
                                       val_with_predictions[prob_column].max()+10**-12,
                                       num=num_grid_points):

    total_reward, average_reward, total_sales, total_time =\
        simulate_threshold(candidate_threshold, best_threshold_at_m[m1], val_with_predictions)

    if average_reward > best_reward:
        best_reward = average_reward
        best_threshold_at_m[m] = candidate_threshold

print("Best threshold at m=" + str(m) + ":", best_threshold_at_m[m])

total_reward_agent, average_reward_agent, \
    total_sales_agent, total_time_agent = simulate_threshold(best_threshold_at_m[m1],
                                                             best_threshold_at_m[m2],
                                                             test_with_predictions)

print()
print("Test set reward with the stopping agent:")
print("Total reward on test:", total_reward_agent)
print("Avg. reward on test:", average_reward_agent)
print("Total sales on test:", total_sales_agent)
print("Total time on test:", total_time_agent)
print()

print("Comparative results:")
print("Sales lost by stopping agent:", total_sales_noagent - total_sales_agent)
print("Time saved by stopping agent (seconds):", total_time_noagent - total_time_agent)
print("Time saved by stopping agent (%):", (total_time_noagent - total_time_agent)/\
                                            total_time_noagent * 100)

Best threshold at m=45: 0.5257815551975125
Best threshold at m=60: 0.5264289506984999

Test set reward with the stopping agent:
Total reward on test: -75.52300000000002
Avg. reward on test: -1.5104600000000006
Total sales on test: 25
Total time on test: 3255.23

Comparative results:
Sales lost by stopping agent: 0
Time saved by stopping agent (seconds): 24.559999999999945
Time saved by stopping agent (%): 0.7488284310885741
